# Book Cover Genre Classifier
Predicting the genre of a book from its cover image using PyTorch.

**Dataset**: 50,000+ Amazon book covers across 30 genres  
**Approach**: Custom CNN baseline → Transfer learning with ResNet50  
**Goal**: Can a model judge a book by its cover?

In [1]:
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.optim as optim
from torchvision import models

## Data Loading
The dataset provides images and labels in separate files, so we need a custom `Dataset` class rather than PyTorch's built-in `ImageFolder`. The class reads each image filename and its genre label from a CSV, maps genres to integer indices, and applies transforms on the fly.

In [2]:
class BookCoverDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file, sep=';')
        self.img_dir = img_dir
        self.transform = transform

        self.categories = sorted(self.data['Category'].unique())
        self.cat_to_idx = {cat: i for i, cat in enumerate(self.categories)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data.iloc[idx]['Filename'])
        image = Image.open(img_name).convert('RGB')
        label = self.cat_to_idx[self.data.iloc[idx]['Category']]

        if self.transform:
            image = self.transform(image)

        return image, label

In [3]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])
train_dataset = BookCoverDataset(
    csv_file='/kaggle/input/datasets/mexwell/book-cover-dataset/book30-listing-train.csv',
    img_dir='/kaggle/input/datasets/mexwell/book-cover-dataset/title30cat/224x224/',
    transform=transform
)

test_dataset = BookCoverDataset(
    csv_file='/kaggle/input/datasets/mexwell/book-cover-dataset/book30-listing-test.csv',
    img_dir='/kaggle/input/datasets/mexwell/book-cover-dataset/title30cat/224x224/',
    transform=transform
)

trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
testloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Categories: {len(train_dataset.categories)}")

Training samples: 51300
Categories: 30


## Baseline: Custom CNN
First we train a simple 3-layer convolutional neural network from scratch to establish a baseline. This gives us a reference point to compare against transfer learning.

Architecture: 3x Conv2d → MaxPool → Dropout → 3x Fully Connected → 30-class output

In [4]:
class BookCoverCNN(nn.Module):
    def __init__(self):
        super(BookCoverCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 30) #30 categories
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
model = BookCoverCNN().to(device)

Using device: cuda


## Training the CNN
Training for 5 epochs with Adam optimizer and CrossEntropyLoss. We expect modest accuracy since we're training from scratch on a hard 30-class problem.

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 50 == 49:
            print(f'Epoch {epoch+1}, Step {i+1}, Loss: {running_loss/50:.3f}')
            running_loss = 0.0

print('Training done!')

Epoch 1, Step 50, Loss: 3.411
Epoch 1, Step 100, Loss: 3.401
Epoch 1, Step 150, Loss: 3.405
Epoch 1, Step 200, Loss: 3.401
Epoch 1, Step 250, Loss: 3.403
Epoch 1, Step 300, Loss: 3.402
Epoch 1, Step 350, Loss: 3.406
Epoch 1, Step 400, Loss: 3.402
Epoch 1, Step 450, Loss: 3.403
Epoch 1, Step 500, Loss: 3.401
Epoch 1, Step 550, Loss: 3.402
Epoch 1, Step 600, Loss: 3.402
Epoch 1, Step 650, Loss: 3.402
Epoch 1, Step 700, Loss: 3.401
Epoch 1, Step 750, Loss: 3.402
Epoch 1, Step 800, Loss: 3.402
Epoch 1, Step 850, Loss: 3.403
Epoch 1, Step 900, Loss: 3.402
Epoch 1, Step 950, Loss: 3.401
Epoch 1, Step 1000, Loss: 3.402
Epoch 1, Step 1050, Loss: 3.402
Epoch 1, Step 1100, Loss: 3.402
Epoch 1, Step 1150, Loss: 3.401
Epoch 1, Step 1200, Loss: 3.401
Epoch 1, Step 1250, Loss: 3.423
Epoch 1, Step 1300, Loss: 3.401
Epoch 1, Step 1350, Loss: 3.397
Epoch 1, Step 1400, Loss: 3.393
Epoch 1, Step 1450, Loss: 3.397
Epoch 1, Step 1500, Loss: 3.388
Epoch 1, Step 1550, Loss: 3.385
Epoch 1, Step 1600, Loss: 3.

## CNN Evaluation
Testing on the held-out test set. For context, random guessing on 30 classes = 3.3% accuracy.

In [6]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')

Accuracy: 13.84%


## Transfer Learning with ResNet50
Instead of training from scratch, we load ResNet50 pretrained on ImageNet (1.2M images, 1000 classes). We freeze all layers and replace only the final fully connected layer with a 30-class output head.

**Why transfer learning?** ResNet50 already knows how to detect edges, textures, shapes, and color patterns. We just need it to learn what those patterns mean for book genres.

In [7]:
# Load pretrained ResNet50
model = models.resnet50(weights='IMAGENET1K_V1')

# Freeze all layers except the last one
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer for 30 categories
model.fc = nn.Linear(model.fc.in_features, 30)

model = model.to(device)
print("ResNet50 loaded!")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 175MB/s]


ResNet50 loaded!


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) 

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 50 == 49:
            print(f'Epoch {epoch+1}, Step {i+1}, Loss: {running_loss/50:.3f}')
            running_loss = 0.0

print('Training done!')

Epoch 1, Step 50, Loss: 3.484
Epoch 1, Step 100, Loss: 3.203
Epoch 1, Step 150, Loss: 3.142
Epoch 1, Step 200, Loss: 3.053
Epoch 1, Step 250, Loss: 3.017
Epoch 1, Step 300, Loss: 2.971
Epoch 1, Step 350, Loss: 2.964
Epoch 1, Step 400, Loss: 2.982
Epoch 1, Step 450, Loss: 2.963
Epoch 1, Step 500, Loss: 2.983
Epoch 1, Step 550, Loss: 2.973
Epoch 1, Step 600, Loss: 2.958
Epoch 1, Step 650, Loss: 2.968
Epoch 1, Step 700, Loss: 2.978
Epoch 1, Step 750, Loss: 3.022
Epoch 1, Step 800, Loss: 2.905
Epoch 1, Step 850, Loss: 2.977
Epoch 1, Step 900, Loss: 2.911
Epoch 1, Step 950, Loss: 2.941
Epoch 1, Step 1000, Loss: 2.939
Epoch 1, Step 1050, Loss: 2.949
Epoch 1, Step 1100, Loss: 3.010
Epoch 1, Step 1150, Loss: 2.961
Epoch 1, Step 1200, Loss: 2.918
Epoch 1, Step 1250, Loss: 2.981
Epoch 1, Step 1300, Loss: 2.919
Epoch 1, Step 1350, Loss: 2.892
Epoch 1, Step 1400, Loss: 2.947
Epoch 1, Step 1450, Loss: 2.940
Epoch 1, Step 1500, Loss: 2.842
Epoch 1, Step 1550, Loss: 2.907
Epoch 1, Step 1600, Loss: 2.

## Full Fine-Tuning
After training only the final layer, we unfreeze all layers and continue training with a much lower learning rate (1e-4). This allows the entire network to adapt to our specific task while preserving the pretrained features.

In [9]:
for param in model.parameters():
    param.requires_grad = True

# Lower learning rate for fine-tuning
optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(5):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 50 == 49:
            print(f'Epoch {epoch+1}, Step {i+1}, Loss: {running_loss/50:.3f}')
            running_loss = 0.0

print('Fine-tuning done!')

Epoch 1, Step 50, Loss: 2.615
Epoch 1, Step 100, Loss: 2.744
Epoch 1, Step 150, Loss: 2.722
Epoch 1, Step 200, Loss: 2.780
Epoch 1, Step 250, Loss: 2.636
Epoch 1, Step 300, Loss: 2.588
Epoch 1, Step 350, Loss: 2.590
Epoch 1, Step 400, Loss: 2.596
Epoch 1, Step 450, Loss: 2.593
Epoch 1, Step 500, Loss: 2.529
Epoch 1, Step 550, Loss: 2.557
Epoch 1, Step 600, Loss: 2.587
Epoch 1, Step 650, Loss: 2.563
Epoch 1, Step 700, Loss: 2.554
Epoch 1, Step 750, Loss: 2.454
Epoch 1, Step 800, Loss: 2.537
Epoch 1, Step 850, Loss: 2.496
Epoch 1, Step 900, Loss: 2.485
Epoch 1, Step 950, Loss: 2.552
Epoch 1, Step 1000, Loss: 2.443
Epoch 1, Step 1050, Loss: 2.483
Epoch 1, Step 1100, Loss: 2.492
Epoch 1, Step 1150, Loss: 2.484
Epoch 1, Step 1200, Loss: 2.469
Epoch 1, Step 1250, Loss: 2.494
Epoch 1, Step 1300, Loss: 2.491
Epoch 1, Step 1350, Loss: 2.433
Epoch 1, Step 1400, Loss: 2.419
Epoch 1, Step 1450, Loss: 2.473
Epoch 1, Step 1500, Loss: 2.429
Epoch 1, Step 1550, Loss: 2.403
Epoch 1, Step 1600, Loss: 2.

In [10]:
# Keep fine-tuning for more epochs
optimizer = optim.Adam(model.parameters(), lr=0.00005)  # even lower lr

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 50 == 49:
            print(f'Epoch {epoch+1}, Step {i+1}, Loss: {running_loss/50:.3f}')
            running_loss = 0.0

print('Done!')

Epoch 1, Step 50, Loss: 0.392
Epoch 1, Step 100, Loss: 0.295
Epoch 1, Step 150, Loss: 0.270
Epoch 1, Step 200, Loss: 0.299
Epoch 1, Step 250, Loss: 0.270
Epoch 1, Step 300, Loss: 0.256
Epoch 1, Step 350, Loss: 0.291
Epoch 1, Step 400, Loss: 0.274
Epoch 1, Step 450, Loss: 0.280
Epoch 1, Step 500, Loss: 0.281
Epoch 1, Step 550, Loss: 0.254
Epoch 1, Step 600, Loss: 0.285
Epoch 1, Step 650, Loss: 0.261
Epoch 1, Step 700, Loss: 0.236
Epoch 1, Step 750, Loss: 0.262
Epoch 1, Step 800, Loss: 0.272
Epoch 1, Step 850, Loss: 0.270
Epoch 1, Step 900, Loss: 0.262
Epoch 1, Step 950, Loss: 0.305
Epoch 1, Step 1000, Loss: 0.285
Epoch 1, Step 1050, Loss: 0.266
Epoch 1, Step 1100, Loss: 0.265
Epoch 1, Step 1150, Loss: 0.275
Epoch 1, Step 1200, Loss: 0.320
Epoch 1, Step 1250, Loss: 0.260
Epoch 1, Step 1300, Loss: 0.282
Epoch 1, Step 1350, Loss: 0.268
Epoch 1, Step 1400, Loss: 0.277
Epoch 1, Step 1450, Loss: 0.265
Epoch 1, Step 1500, Loss: 0.241
Epoch 1, Step 1550, Loss: 0.259
Epoch 1, Step 1600, Loss: 0.

## ResNet Evaluation

In [11]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')

Accuracy: 28.35%
